На базе баскетбольных матчей добейтесь средней абсолютной ошибки 17 и менее очков.

### Подготовка  

In [1]:
# Загрузка из google облака
import gdown
gdown.download('https://storage.yandexcloud.net/aiueducation/Content/base/l10/basketball.csv', None, quiet=True)

# Библиотека для работы с базами
import pandas as pd
df = pd.read_csv('basketball.csv', encoding= 'cp1251', sep=';', header=0, index_col=0) # Загружаем базу
df.head()

,TOTAL,info,Ком. 1,Ком. 2,Минута,Общая минута,Секунда,fcount,ftime
0,"98,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,0.0,1,1.0,30,81,90.0
1,"100,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,1,1.0,45,81,105.0
2,"99,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,2,2.0,0,81,120.0
3,"98,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,2,2.0,30,81,150.0
4,"95,5",4081445 Новая Зеландия. Женщины. WBC. Регулярн...,2,2.0,3,3.0,0,81,180.0


Извлекаем текстовые данные из колонки `info` таблицы, помещаем в переменную `data_text`. Выводим длину списка:

In [2]:
data_text = df['info'].values #

len(data_text) #

52450

Задаем максимальное кол-во слов в словаре, помещаем в переменную все символы, которые хотим вычистить из текста.

 Токенизируем текстовые данные:

In [3]:
# Импортируем токенайзер
from tensorflow.keras.preprocessing.text import Tokenizer

maxWordsCount = 5000

sim_for_del='!"#$%&()*+,-./:;<=>?@[\\]^_`{|}~\t\n'

tokenizer = Tokenizer (num_words=maxWordsCount,
                       filters=sim_for_del,
                       lower=True,
                       split=' ',
                       oov_token='unknown',
                       char_level=False)

tokenizer.fit_on_texts(data_text)

In [4]:
# Переводим в Эмбеддинг пространство
Sequences = tokenizer.texts_to_sequences(data_text)

# Вариант  Bag of Words
xBOW_text = tokenizer.sequences_to_matrix(Sequences)

Преобразуем данные в numpy, подготовим наборы для обучения:

In [5]:
# Библиотека работы с массивами
import numpy as np

xTrain = np.array(df[['Ком. 1','Ком. 2', 'Минута', 'Секунда','ftime']].astype('int'))
yTrain = np.array(df['fcount'].astype('int'))

In [6]:
print(xTrain.shape)
print(yTrain.shape)
print(xBOW_text.shape)

(52450, 5)
(52450,)
(52450, 5000)


In [7]:
# Функция по проверке ошибки

def check_MAE_predictl_DubbleInput (model,
                                    x_data,
                                    x_data_text,
                                    y_data_not_scaled,
                                    plot=False):

  mae = 0 # Инициализируем начальное значение ошибки
  y_pred = (model.predict([x_data,x_data_text])).squeeze()

  for n in range (0,len(x_data)):
    mae += abs(y_data_not_scaled[n] - y_pred[n]) # Увеличиваем значение ошибки для текущего элемента
  mae /= len(x_data) # Считаем среднее значение
  print('Среднаяя абслолютная ошибка {:.3f} очков это {:.3f}% от общей выборки в {} игры'.format(mae, (mae/y_data_not_scaled.mean(axis=0))*100,len(x_data)))

  if plot:
     plt.scatter(y_data_not_scaled, y_pred)
     plt.xlabel('Правильные значение')
     plt.ylabel('Предсказания')
     plt.axis('equal')
     plt.xlim(plt.xlim())
     plt.ylim(plt.ylim())
     plt.plot([0, 250], [0, 250])
     plt.show()

In [8]:
def on_epoch_end_custom(epoch, logs=None):
    check_MAE_predictl_DubbleInput(model_final_scaled,xTrain_scaled,xBOW_text,yTrain,plot=True)

In [21]:
# Подготовка данных

import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, concatenate, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LambdaCallback, ReduceLROnPlateau
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import re

df = pd.read_csv('basketball.csv', encoding='cp1251', sep=';', header=0, index_col=0)

# Числовые признаки
numeric_features = ['Ком. 1', 'Ком. 2', 'Минута', 'Секунда']
xTrain_num = df[numeric_features].values.astype('float32')

# Очистка текста от идентификатора матча в начале
def clean_text(text):
    text = str(text)
    text = re.sub(r'^\d+\s+', '', text)
    return text

df['info_cleaned'] = df['info'].apply(clean_text)

# Токенизация
max_words = 5000
tokenizer = Tokenizer(num_words=max_words, lower=True, oov_token='unknown')
tokenizer.fit_on_texts(df['info_cleaned'])
sequences = tokenizer.texts_to_sequences(df['info_cleaned'])
xBOW_text = tokenizer.sequences_to_matrix(sequences)

# Целевая переменная
yTrain = df['fcount'].values.astype('float32')

print(f"Числовые данные: {xTrain_num.shape}")
print(f"Текстовые данные: {xBOW_text.shape}")
print(f"Целевая переменная: {yTrain.shape}")

Числовые данные: (52450, 4)
Текстовые данные: (52450, 5000)
Целевая переменная: (52450,)


In [22]:
# Масштабирование и разбиение

scaler_X = StandardScaler()
xTrain_num_scaled = scaler_X.fit_transform(xTrain_num)

scaler_y = StandardScaler()
yTrain_scaled = scaler_y.fit_transform(yTrain.reshape(-1, 1)).flatten()

X_train_num, X_val_num, X_train_text, X_val_text, y_train, y_val = train_test_split(
    xTrain_num_scaled, xBOW_text, yTrain_scaled, test_size=0.2, random_state=42
)

print(f"Обучающая выборка: {X_train_num.shape[0]} samples")
print(f"Валидационная выборка: {X_val_num.shape[0]} samples")

Обучающая выборка: 41960 samples
Валидационная выборка: 10490 samples


In [23]:
# Модель

input_num = Input(shape=(X_train_num.shape[1],))
input_text = Input(shape=(X_train_text.shape[1],))

x = concatenate([input_num, input_text])
x = Dense(128, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.3)(x)
x = Dense(64, activation='relu')(x)
x = Dense(32, activation='relu')(x)
output = Dense(1, activation='linear')(x)

model = Model(inputs=[input_num, input_text], outputs=output)
model.compile(optimizer=Adam(learning_rate=0.001), loss='mse', metrics=['mae'])

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 4)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, 5000)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 5004)      │          0 │ input_layer_2[0]… │
│ (Concatenate)       │                   │            │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 128)       │    640,640 │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128)       │        512 │ dense_4[0][0]     │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_1 (Dropout) │ (None, 128)       │          0 │ batch_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_5 (Dense)     │ (None, 64)        │      8,256 │ dropout_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 32)        │      2,080 │ dense_5[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 1)         │         33 │ dense_6[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 651,521 (2.49 MB)

 Trainable params: 651,265 (2.48 MB)

 Non-trainable params: 256 (1.00 KB)

In [24]:
# Обучение

def custom_callback(epoch, logs=None):
    pred_scaled = model.predict([X_train_num, X_train_text], verbose=0)
    pred_real = scaler_y.inverse_transform(pred_scaled).flatten()
    y_train_real = scaler_y.inverse_transform(y_train.reshape(-1, 1)).flatten()
    mae = np.mean(np.abs(pred_real - y_train_real))
    print(f"  MAE на обучении: {mae:.3f} фолов")

callback = LambdaCallback(on_epoch_end=custom_callback)

history = model.fit(
    [X_train_num, X_train_text], y_train,
    validation_data=([X_val_num, X_val_text], y_val),
    epochs=30, batch_size=64, verbose=1,
    callbacks=[callback]
)

Epoch 1/30
656/656 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - loss: 0.3353 - mae: 0.3804  MAE на обучении: 4.148 фолов
656/656 ━━━━━━━━━━━━━━━━━━━━ 15s 18ms/step - loss: 0.1726 - mae: 0.2731 - val_loss: 0.0647 - val_mae: 0.1698
Epoch 2/30
655/656 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0713 - mae: 0.1818  MAE на обучении: 2.521 фолов
656/656 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - loss: 0.0658 - mae: 0.1723 - val_loss: 0.0314 - val_mae: 0.1036
Epoch 3/30
641/656 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0555 - mae: 0.1532  MAE на обучении: 2.444 фолов
656/656 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - loss: 0.0537 - mae: 0.1504 - val_loss: 0.0321 - val_mae: 0.1012
Epoch 4/30
654/656 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 0.0479 - mae: 0.1407  MAE на обучении: 2.696 фолов
656/656 ━━━━━━━━━━━━━━━━━━━━ 8s 13ms/step - loss: 0.0479 - mae: 0.1398 - val_loss: 0.0368 - val_mae: 0.1129
Epoch 5/30
656/656 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 0.0444 - mae: 0.1315  MAE на обучении: 2.139 фолов
656/656 ━━━━

In [25]:
from sklearn.metrics import mean_absolute_error

pred_scaled = model.predict([xTrain_num_scaled, xBOW_text], verbose=0)
pred_real = scaler_y.inverse_transform(pred_scaled).flatten()
mae = mean_absolute_error(yTrain, pred_real)

print(f"MAE: {mae:.3f} фолов")
print(f"Целевой порог: 17 фолов")

if mae <= 17:
    print("Задание выполнено")
else:
    print("Задание не выполнено")

MAE: 1.653 фолов
Целевой порог: 17 фолов
Задание выполнено
